In [1]:
reference_chart = {
    "Leukocytes": {
        "Negative": [254, 248, 188],
        "15 - Trace": [229, 218, 174],
        "75 - Small": [206, 157, 149],
        "125 - Moderate": [166, 120, 153],
        "500 - Large": [110, 83, 124]
    },
    "Nitrite": {
        "Negative": [253, 250, 222],
        "Positive (+)": [251, 220, 218],
        "Positive (++)": [247, 181, 200],
        "Positive (+++)": [238, 78, 130]
    },
    "Urobilinogen": {
        "3.2 - Normal": [254, 211, 174],
        "16 - Normal": [248, 168, 133],
        "32 - +": [242, 130, 145],
        "64 - ++": [230, 111, 128],
        "128 - +++": [238, 78, 130]
    },
    "Protein": {
        "Negative": [222, 229, 125],
        "Trace - +-": [187, 215, 106],
        "0.3 +": [172, 212, 130],
        "1 = ++": [119, 189, 151],
        "3 = +++": [94, 178, 169],
        ">= 20 = ++++": [0, 148, 149]
    },
    "pH": {
        "5.0": [245, 139, 79],
        "6.0": [249, 165, 85],
        "6.5": [253, 195, 109],
        "7.0": [208, 189, 98],
        "7.5": [136, 148, 85],
        "8.0": [86, 173, 145],
        "8.5": [0, 127, 129]
    },
    "Blood": {
        "Negative": [250, 174, 76],
        "10 - Trace": [207, 161, 65],
        "25 - Small": [161, 156, 84],
        "80 - Moderate": [116, 156, 122],
        "200 - Large": [69, 128, 108]
    },
    "Sp. Gravity": {
        "1.000": [2, 113, 126],
        "1.005": [76, 117, 102],
        "1.010": [123, 136, 105],
        "1.015": [155, 141, 58],
        "1.020": [175, 161, 52],
        "1.025": [197, 167, 48],
        "1.030": [210, 171, 43]
    },
    "Ketone": {
        "Negative": [251, 187, 149],
        "0.5 - Trace": [246, 158, 137],
        "1.5 - Small": [243, 131, 140],
        "4.0 - Moderate": [201, 88, 116],
        "8.0 - Large": [150, 58, 102],
        "16.0 - Large": [120, 41, 90]
    },
    "Bilirubin": {
        "Negative (Baseline)": [253, 250, 222],
        "Negative (Alt)": [253, 223, 144],
        "17 - Small": [251, 187, 131],
        "50 - Moderate": [208, 146, 136],
        "100 - Large": [171, 127, 131]
    },
    "Glucose": {
        "Negative (Blue)": [111, 203, 220],
        "Negative (Green)": [141, 208, 187],
        "5 - Trace": [152, 207, 148],
        "15 - +": [139, 171, 106],
        "30 - ++": [164, 129, 67],
        "60 - +++": [157, 105, 37],
        "110 - ++++": [136, 89, 41]
    }
}

In [2]:
import colour
def rgb_to_lab(rgb_array):
    xyz = colour.sRGB_to_XYZ(rgb_array)
    lab = colour.XYZ_to_Lab(xyz)
    return lab

In [3]:
import numpy as np
def get_closest_match(extracted_rgb, pad_name, chart):
    # # Convert extracted RGB to LAB
    lab_pixel = rgb_to_lab(extracted_rgb)
    #
    best_label = ""
    min_diff = float('inf')
    #
    for label, ref_rgb in chart[pad_name].items():
        # Convert reference RGB to LAB
        ref_rgb_pixels = np.array([[ref_rgb]], dtype=np.uint8)
        ref_lab = rgb_to_lab(ref_rgb_pixels)

        # Calculate Delta E (CIEDE2000 standard)
        diff = colour.delta_E(lab_pixel, ref_lab, method = 'CIE2000')

        if diff < min_diff:
            min_diff = diff
            best_label = label

    return best_label

# --- 2. IMAGE PROCESSING ---
pad_names =["Leukocytes", "Nitrite", "Urobilinogen", "Protein", "pH", 
                 "Blood", "Sp. Gravity", "Ketone", "Bilirubin", "Glucose"]


In [7]:
import cv2
import cv2.aruco as aruco
import numpy as np

# 1. Define the dictionary and detector parameters
# DICT_6X6_250 contains 250 markers with a 6x6 internal grid
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_6X6_250)
parameters = aruco.DetectorParameters()

# Initialize the webcam
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
while True:
    ret, frame = cap.read() # Frame contains the image from webcam
    if not ret:
        break
    warped = frame.copy()
    # 2. Detect the markers
    corners, ids, rejected = aruco.detectMarkers(frame, aruco_dict, parameters=parameters)

    # 3. Draw the results
    if ids is not None:
        src_pts = corners[0].reshape(4,2).astype(np.float32)
        dst_pts =np.float32([[0, 0], [200, 0], [200, 200], [0, 200]])
        matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
        warped = cv2.warpPerspective(frame, matrix, (800, 600))

        # Calculate x and y coordinates of all Traces
        x1 = src_pts[0][0]
        x2 = src_pts[1][0]
        y1 = src_pts[0][1]
        y2 = src_pts[1][1]

        cm_per_pixel = 4.5/np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
        corrected_distance = cm_per_pixel*np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
        x_distance_cm = 4.5+1.4+0.7/2
        x_distance = x_distance_cm/cm_per_pixel
        x_peestick = int(x1 + x_distance)
        x_peestick_cm = (x_peestick-x1)*cm_per_pixel

        y_scaling = 1.5
        vertical_distance_cm =  0.4+0.7 #Spacing + 2 halves of squares
        vertical_distance_pixel = round(vertical_distance_cm/cm_per_pixel)

        leukocytes_y = int(y_scaling*(10+0*vertical_distance_pixel))
        nitrite_y = int(y_scaling*(10+1*vertical_distance_pixel))
        urobilinogen_y = int(y_scaling*(10+2*vertical_distance_pixel))
        protein_y = int(y_scaling*(10+3*vertical_distance_pixel))
        pH_y = int(y_scaling*(10+4*vertical_distance_pixel))
        blood_y = int(y_scaling*(10+5*vertical_distance_pixel))
        sp_gravity_y = int(y_scaling*(10+6*vertical_distance_pixel))
        ketone_y = int(y_scaling*(10+7*vertical_distance_pixel))
        bilirubin_y = int(y_scaling*(10+8*vertical_distance_pixel))
        glucose_y = int(y_scaling*(10+9*vertical_distance_pixel))

        # Transform x and y coordinates to warped image
        leukocytes_point = np.array([[x_peestick, leukocytes_y]], dtype=np.float32)
        point = np.array([leukocytes_point])  # shape should be (1,1,2)
        warped_point = cv2.perspectiveTransform(point, matrix)
        cv2.circle(warped, tuple(warped_point[0][0].astype(int)), radius=5, color=(0,0,255), thickness=-1)

        rgb_frame = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)

        extracted_colours = []

        # Check that we hit the correct markers on the pee-stick
        rgb_leukocytes = [rgb_frame[leukocytes_y,x_peestick]]
        extracted_colours.append(rgb_leukocytes)

        rgb_nitrite = rgb_frame[nitrite_y, x_peestick]
        extracted_colours.append(rgb_nitrite)

        rgb_urobilinogen = rgb_frame[urobilinogen_y, x_peestick]
        extracted_colours.append(rgb_urobilinogen)

        rgb_protein = rgb_frame[protein_y, x_peestick]
        extracted_colours.append(rgb_protein)

        rgb_pH = rgb_frame[pH_y, x_peestick]
        extracted_colours.append(rgb_pH)

        rgb_blood = rgb_frame[blood_y, x_peestick]
        extracted_colours.append(rgb_blood)

        rgb_sp_gravity = rgb_frame[sp_gravity_y, x_peestick]
        extracted_colours.append(rgb_sp_gravity)

        rgb_ketone = rgb_frame[ketone_y, x_peestick]
        extracted_colours.append(rgb_ketone)

        rgb_bilirubin = rgb_frame[bilirubin_y, x_peestick]
        extracted_colours.append(rgb_bilirubin)

        rgb_glucose = rgb_frame[glucose_y, x_peestick]
        extracted_colours.append(rgb_glucose)

        aruco.drawDetectedMarkers(frame, corners, ids)

        print("--- Extracted Pad Colors (RGB) ---")
        pad_names =["Leukocytes", "Nitrite", "Urobilinogen", "Protein", "pH",
                 "Blood", "Sp. Gravity", "Ketone", "Bilirubin", "Glucose"]

        for i, RGB in enumerate(extracted_colours):
            name = pad_names[i] if i < len(pad_names) else f"Unknown Pad {i}"
            result = get_closest_match(RGB, name, reference_chart)
            print(f"{name:.<15}: {result:.<15}. RGB:{RGB}")

        # Check that we hit the correct markers on the pee-stick
        cv2.circle(warped, (round(x_peestick), leukocytes_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), nitrite_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), urobilinogen_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), protein_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), pH_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), blood_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), sp_gravity_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), ketone_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), bilirubin_y), 6, (0,0,255), -2)
        cv2.circle(warped, (round(x_peestick), glucose_y), 6, (0,0,255), -2)

    cv2.imshow('Pee-Stick Detection', warped)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

--- Extracted Pad Colors (RGB) ---
Leukocytes.....: 75 - Small...... RGB:[array([229, 225, 226], dtype=uint8)]
Nitrite........: Positive (++)... RGB:[223 222 226]
Urobilinogen...: 32 - +.......... RGB:[226 222 227]
Protein........: 1 = ++.......... RGB:[227 223 226]
pH.............: 5.0............. RGB:[227 223 225]
Blood..........: Negative........ RGB:[225 222 224]
Sp. Gravity....: 1.030........... RGB:[225 223 223]
Ketone.........: 1.5 - Small..... RGB:[227 222 224]
Bilirubin......: 100 - Large..... RGB:[227 219 220]
Glucose........: 60 - +++........ RGB:[226 222 223]
--- Extracted Pad Colors (RGB) ---
Leukocytes.....: 125 - Moderate.. RGB:[array([223, 220, 222], dtype=uint8)]
Nitrite........: Positive (++)... RGB:[221 221 224]
Urobilinogen...: 32 - +.......... RGB:[224 223 227]
Protein........: 1 = ++.......... RGB:[226 223 226]
pH.............: 6.0............. RGB:[220 221 223]
Blood..........: Negative........ RGB:[221 222 224]
Sp. Gravity....: 1.030........... RGB:[224 222 223